### Infinity Throughput Benchmark Notebook

这个 notebook 参照 `inference_single_Infinity.ipynb` 的加载与推理流程，进行端到端图片生成吞吐测试：
- 模型加载完成后开始计时
- 支持手动设置 `batch_size`（单个大 batch 内连续生成多少张）
- 支持手动设置 warmup 与正式测试的大 batch 次数
- 最终输出每个大 batch 耗时、平均大 batch 耗时、平均单图耗时，以及 IPS（images per second）


In [21]:
import os
import sys
import time
import argparse
import random

import cv2
import numpy as np
import torch
import os.path as osp

# 通过环境变量选择 GPU，例如: GPU_ID=1
GPU_ID = int(os.environ.get("GPU_ID", "5"))
torch.cuda.set_device(GPU_ID)

project_root = '/home/jiaji_lu/AR/VAR-Q'
os.chdir(project_root)
sys.path.append(project_root)

from Infinity.tools.run_infinity import *
from Infinity.tools.run_infinity import _import_dynamic_resolution

############# Configuration File #############
CONFIG_FILE = "/home/jiaji_lu/AR/VAR-Q/temp/Infinity-VAR_Q-8.json"

print(f"[Config] Loading configuration from {CONFIG_FILE}")

var_q_path = '/home/jiaji_lu/AR/VAR-Q/VAR_Q'
if os.path.exists(var_q_path):
    sys.path.append(var_q_path)
    from config_loader import VARQConfig

    try:
        config = VARQConfig(CONFIG_FILE)

        model_config = config.get_model_config()
        quant_config = config.get_quantization_config()
        inference_config = config.get_inference_config()
        batch_config = config.get_batch_processing_config()
        checkpoint_config = config.get_checkpoint_config()

        print("[Config] Configuration loaded successfully!")
        print(f"[Config] Model: {model_config.get('model_type')}")
        print(f"[Config] VAR-Q Quantization: {'enabled' if quant_config.get('enable') else 'disabled'}")
        if quant_config.get('enable'):
            print(f"[Config]   - q_bits: {quant_config.get('q_bits')}")
            print(f"[Config]   - quant_method: {quant_config.get('quant_method')}")
            print(f"[Config]   - qkv_format: {quant_config.get('qkv_format')}")
            print(f"[Config]   - rescale_qk: {quant_config.get('rescale_qk', False)}")

    except Exception as e:
        print(f"[Error] Failed to load configuration: {e}")
        raise
else:
    print(f"[Error] VAR_Q path not found: {var_q_path}")
    raise FileNotFoundError(f"VAR_Q directory not found at {var_q_path}")


[Config] Loading configuration from /home/jiaji_lu/AR/VAR-Q/temp/Infinity-VAR_Q-8.json
[Config] Configuration loaded successfully!
[Config] Model: infinity_8b
[Config] VAR-Q Quantization: disabled


In [22]:
############# Create args from configuration #############
model_type = model_config.get('model_type', 'infinity_2b')

if model_type == "infinity_2b":
    vae_type = 32
    apply_spatial_patchify = 0
    checkpoint_type = "torch"
elif model_type == "infinity_8b":
    vae_type = 14
    apply_spatial_patchify = 1
    checkpoint_type = "torch_shard"
else:
    vae_type = 32
    apply_spatial_patchify = 0
    checkpoint_type = "torch"

args = argparse.Namespace(
    model_type=model_type,
    pn='1M',
    model_path=checkpoint_config.get('model_path'),
    vae_path=checkpoint_config.get('vae_ckpt'),
    text_encoder_ckpt='/data/boxunxu/Infinity/flan-t5-xl',

    vae_type=vae_type,
    apply_spatial_patchify=apply_spatial_patchify,
    checkpoint_type=checkpoint_type,

    add_lvl_embeding_only_first_block=1,
    use_bit_label=1,
    rope2d_each_sa_layer=1,
    rope2d_normalized_by_hw=2,
    use_scale_schedule_embedding=0,
    sampling_per_bits=1,
    text_channels=2048,
    h_div_w_template=inference_config.get('h_div_w', 1.0),
    use_flex_attn=0,

    cache_dir='/dev/shm',
    seed=inference_config.get('seed', 0),
    bf16=1,
    save_file='tmp.jpg',
    enable_model_cache=0,

    cfg_insertion_layer=0,
    enable_positive_prompt=0,
    cfg=inference_config.get('cfg', 3.0),
    tau=inference_config.get('tau', 0.5),

    enable_quantization=int(quant_config.get('enable', False)),
    q_bits=quant_config.get('q_bits', 8),
    quant_method=quant_config.get('quant_method', 'G_SCALE_HEAD_DIM'),
    qkv_format=quant_config.get('qkv_format', 'BLHc'),
    rescale_qk=int(quant_config.get('rescale_qk', False)),
)

print("[Args] Arguments created from configuration:")
print(f"  - Model: {args.model_type}")
print(f"  - Model path: {args.model_path}")
print(f"  - VAE path: {args.vae_path}")
print(f"  - VAE type: {args.vae_type}")
print(f"  - VAR-Q Quantization: {'Enabled' if args.enable_quantization else 'Disabled'}")
if args.enable_quantization:
    print(f"    * Bits: {args.q_bits}")
    print(f"    * Method: {args.quant_method}")
    print(f"    * Format: {args.qkv_format}")
    print(f"    * Rescale Q/K: {args.rescale_qk}")
print(f"  - CFG: {args.cfg}")
print(f"  - Tau: {args.tau}")
print(f"  - Seed: {args.seed}")


[Args] Arguments created from configuration:
  - Model: infinity_8b
  - Model path: /data/jiaji_lu/Infinity/infinity_8b_weights
  - VAE path: /data/jiaji_lu/Infinity/infinity_vae_d56_f8_14_patchify.pth
  - VAE type: 14
  - VAR-Q Quantization: Disabled
  - CFG: 3.0
  - Tau: 0.5
  - Seed: 0


In [23]:
############# load model #############
print("[Loading tokenizer and text encoder]")
text_tokenizer, text_encoder = load_tokenizer(t5_path=args.text_encoder_ckpt)

print("[Loading VAE]")
vae = load_visual_tokenizer(args)

print("[Loading Infinity]")
infinity = load_transformer(vae, args)

# 预加载动态分辨率模板，避免把一次性初始化记入吞吐统计
if 'dynamic_resolution_h_w' not in globals() or dynamic_resolution_h_w is None:
    dynamic_resolution_h_w, h_div_w_templates = _import_dynamic_resolution()

print("[Model] All modules loaded. Ready for throughput benchmark.")


[Loading tokenizer and text encoder]
[Loading tokenizer and text encoder]


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 10.72it/s]


[Loading VAE]
[Loading Infinity]
[Loading Infinity]
self.codebook_dim: 56, self.add_lvl_embeding_only_first_block: 1,             self.use_bit_label: 1, self.rope2d_each_sa_layer: 1, self.rope2d_normalized_by_hw: 2
self.num_blocks_in_a_chunk=5, depth=40, block_chunks=8

[constructor]  ==== customized_flash_attn=False (using_flash=0/40), fused_mlp=False (fused_mlp=0/40) ==== 
    [Infinity config ] embed_dim=3584, num_heads=28, depth=40, mlp_ratio=4, swiglu=False num_blocks_in_a_chunk=5
    [drop ratios] drop_rate=0.0, drop_path_rate=0.1 (tensor([0.0000, 0.0026, 0.0051, 0.0077, 0.0103, 0.0128, 0.0154, 0.0179, 0.0205,
        0.0231, 0.0256, 0.0282, 0.0308, 0.0333, 0.0359, 0.0385, 0.0410, 0.0436,
        0.0462, 0.0487, 0.0513, 0.0538, 0.0564, 0.0590, 0.0615, 0.0641, 0.0667,
        0.0692, 0.0718, 0.0744, 0.0769, 0.0795, 0.0821, 0.0846, 0.0872, 0.0897,
        0.0923, 0.0949, 0.0974, 0.1000]))

[you selected Infinity with model_kwargs={'depth': 40, 'embed_dim': 3584, 'num_heads': 28, 'd

In [24]:
############# throughput benchmark (true KV batch) #############
# 固定 prompt 做吞吐测试
prompt = "a tiny astronaut hatching from an egg on the moon"

cfg = inference_config.get('cfg', 3.0)
tau = inference_config.get('tau', 0.5)
h_div_w = inference_config.get('h_div_w', 1.0)
seed = inference_config.get('seed', 0)

# 兼容配置里的历史拼写: enable_positivee_prompt
enable_positive_prompt = inference_config.get(
    'enable_positive_prompt',
    inference_config.get('enable_positivee_prompt', 0)
)

# ===================== 手动设置区域 =====================
# 真 batch：一次调用 autoregressive_infer_cfg 时的 B
batch_size = 1
# warmup 的大 batch 次数
warmup_big_batches = 1
# 正式计时的大 batch 次数（最终在这些大 batch 上取平均）
test_big_batches = 3
# =======================================================

h_div_w_template_ = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - h_div_w))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template_][args.pn]['scales']
scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]

print("[Benchmark] Settings:")
print(f"  - prompt: {prompt}")
print(f"  - cfg: {cfg}")
print(f"  - tau: {tau}")
print(f"  - h_div_w: {h_div_w}")
print(f"  - seed: {seed}")
print(f"  - batch_size (images per big batch): {batch_size}")
print(f"  - warmup_big_batches: {warmup_big_batches}")
print(f"  - test_big_batches: {test_big_batches}")

import torch.nn.functional as F

cfg_list = cfg if isinstance(cfg, list) else [cfg] * len(scale_schedule)
tau_list = tau if isinstance(tau, list) else [tau] * len(scale_schedule)

@torch.no_grad()
def encode_prompt_batch(prompts, use_positive_prompt=False):
    if use_positive_prompt:
        prompts = [aug_with_positive_prompt(p) for p in prompts]

    tokens = text_tokenizer(
        text=prompts,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = tokens.input_ids.cuda(non_blocking=True)
    mask = tokens.attention_mask.cuda(non_blocking=True)

    text_features = text_encoder(input_ids=input_ids, attention_mask=mask)['last_hidden_state'].float()
    lens = mask.sum(dim=-1).tolist()
    cu_seqlens_k = F.pad(mask.sum(dim=-1).to(dtype=torch.int32).cumsum_(0), (1, 0))
    Ltext = max(lens)

    kv_compact = []
    for len_i, feat_i in zip(lens, text_features.unbind(0)):
        kv_compact.append(feat_i[:len_i])
    kv_compact = torch.cat(kv_compact, dim=0)

    return kv_compact, lens, cu_seqlens_k, Ltext

@torch.no_grad()
def run_one_true_batch(cur_seed: int, current_batch_size: int):
    prompts = [prompt] * current_batch_size
    text_cond_tuple = encode_prompt_batch(prompts, use_positive_prompt=bool(enable_positive_prompt))

    with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16, cache_enabled=True):
        _, _, img_batch = infinity.autoregressive_infer_cfg(
            vae=vae,
            scale_schedule=scale_schedule,
            label_B_or_BLT=text_cond_tuple,
            g_seed=cur_seed,
            B=current_batch_size,
            negative_label_B_or_BLT=None,
            force_gt_Bhw=None,
            cfg_sc=3,
            cfg_list=cfg_list,
            tau_list=tau_list,
            top_k=900,
            top_p=0.97,
            returns_vemb=1,
            ratio_Bl1=None,
            gumbel=0,
            norm_cfg=False,
            cfg_exp_k=0.0,
            cfg_insertion_layer=[args.cfg_insertion_layer],
            vae_type=args.vae_type,
            softmax_merge_topk=-1,
            ret_img=True,
            trunk_scale=1000,
            gt_leak=0,
            gt_ls_Bl=None,
            inference_mode=True,
            sampling_per_bits=args.sampling_per_bits,
        )
    return img_batch

# Warmup（按大 batch）
print("\n[Benchmark] Warmup start...")
for i in range(warmup_big_batches):
    warmup_seed = seed + i
    _ = run_one_true_batch(warmup_seed, batch_size)
    torch.cuda.synchronize()
print("[Benchmark] Warmup done.")

# Timed runs（按大 batch）
print("\n[Benchmark] Timed test start...")
big_batch_times = []
peak_alloc_bytes_list = []
peak_reserved_bytes_list = []
last_image_batch = None

base_test_seed = seed + warmup_big_batches

for i in range(test_big_batches):
    cur_seed = base_test_seed + i

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()

    try:
        last_image_batch = run_one_true_batch(cur_seed, batch_size)
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError as oom:
        torch.cuda.empty_cache()
        raise RuntimeError(
            f"CUDA OOM at big-batch {i + 1}. 当前 batch_size={batch_size} 过大，请降低后重试。"
        ) from oom

    t1 = time.perf_counter()
    dt = t1 - t0
    big_batch_times.append(dt)

    peak_alloc_bytes = torch.cuda.max_memory_allocated()
    peak_reserved_bytes = torch.cuda.max_memory_reserved()
    peak_alloc_bytes_list.append(peak_alloc_bytes)
    peak_reserved_bytes_list.append(peak_reserved_bytes)

    peak_alloc_mib = peak_alloc_bytes / (1024 ** 2)
    peak_reserved_mib = peak_reserved_bytes / (1024 ** 2)

    print(
        f"  - big-batch {i + 1:02d}/{test_big_batches}: {dt:.4f} s ({batch_size} images) | "
        f"peak_alloc={peak_alloc_mib:.1f} MiB, peak_reserved={peak_reserved_mib:.1f} MiB"
    )

total_test_time = float(np.sum(big_batch_times))
total_images = int(test_big_batches * batch_size)

avg_big_batch_time = float(np.mean(big_batch_times))
avg_image_time = total_test_time / total_images if total_images > 0 else float('inf')
ips = total_images / total_test_time if total_test_time > 0 else float('inf')

max_peak_alloc_bytes = max(peak_alloc_bytes_list) if peak_alloc_bytes_list else 0
max_peak_reserved_bytes = max(peak_reserved_bytes_list) if peak_reserved_bytes_list else 0

max_peak_alloc_mib = max_peak_alloc_bytes / (1024 ** 2)
max_peak_reserved_mib = max_peak_reserved_bytes / (1024 ** 2)
max_peak_alloc_gib = max_peak_alloc_bytes / (1024 ** 3)
max_peak_reserved_gib = max_peak_reserved_bytes / (1024 ** 3)

print("\n[Benchmark] Result:")
print(f"  - Average big-batch time: {avg_big_batch_time:.4f} s/batch")
print(f"  - Average inference time: {avg_image_time:.4f} s/image")
print(f"  - IPS: {ips:.4f} images/s")
print(f"  - Total tested images: {total_images}")
print(f"  - Total test time: {total_test_time:.4f} s")
print(f"  - Max peak memory (allocated): {max_peak_alloc_mib:.1f} MiB ({max_peak_alloc_gib:.2f} GiB)")
print(f"  - Max peak memory (reserved): {max_peak_reserved_mib:.1f} MiB ({max_peak_reserved_gib:.2f} GiB)")

# 可选：保存最后一个 batch 的第 1 张图，确认输出正常（不计入计时）
out_file = f"Benchmark/outputs/Infinity/throughput-last-{args.quant_method}-{args.q_bits}b.jpg"
os.makedirs(osp.dirname(osp.abspath(out_file)), exist_ok=True)
cv2.imwrite(out_file, last_image_batch[0].cpu().numpy())
print(f"  - Last image saved to: {osp.abspath(out_file)}")


[Benchmark] Settings:
  - prompt: a tiny astronaut hatching from an egg on the moon
  - cfg: 3.0
  - tau: 0.5
  - h_div_w: 1
  - seed: 0
  - batch_size (images per big batch): 1
  - warmup_big_batches: 1
  - test_big_batches: 3

[Benchmark] Warmup start...


/tmp/ipykernel_3813814/3182503005.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True, dtype=torch.bfloat16, cache_enabled=True):


[Benchmark] Warmup done.

[Benchmark] Timed test start...
  - big-batch 01/3: 3.2307 s (1 images) | peak_alloc=37900.1 MiB, peak_reserved=55618.0 MiB
  - big-batch 02/3: 3.2174 s (1 images) | peak_alloc=37903.1 MiB, peak_reserved=55618.0 MiB
  - big-batch 03/3: 3.2187 s (1 images) | peak_alloc=37903.1 MiB, peak_reserved=55618.0 MiB

[Benchmark] Result:
  - Average big-batch time: 3.2223 s/batch
  - Average inference time: 3.2223 s/image
  - IPS: 0.3103 images/s
  - Total tested images: 3
  - Total test time: 9.6668 s
  - Max peak memory (allocated): 37903.1 MiB (37.01 GiB)
  - Max peak memory (reserved): 55618.0 MiB (54.31 GiB)
  - Last image saved to: /home/jiaji_lu/AR/VAR-Q/Benchmark/outputs/Infinity/throughput-last-G_SCALE_HEAD_DIM-8b.jpg
